In [2]:
from pathlib import Path
import gc

import numpy as np
import scipy.sparse as sp
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
)

# Support running Jupyter from either the repository root or ml/.
DATA_ROOT_CANDIDATES = [
    Path.cwd() / 'ml' / 'feature_datasets',
    Path.cwd() / 'ml' / 'feature_dataset',
    Path.cwd() / 'feature_datasets',
    Path.cwd() / 'feature_dataset',
    Path.cwd().parent / 'ml' / 'feature_datasets',
    Path.cwd().parent / 'ml' / 'feature_dataset',
]
DATA_ROOT = next((path for path in DATA_ROOT_CANDIDATES if path.is_dir()), None)
if DATA_ROOT is None:
    searched = '\n'.join(str(path) for path in DATA_ROOT_CANDIDATES)
    raise FileNotFoundError(f'Could not find the feature dataset folder. Searched:\n{searched}')

best_params_by_n = {
    2: {'max_features': 'sqrt', 'n_jobs': 4, 'random_state': 42,
        'max_depth': 40, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200},
    3: {'max_features': 'sqrt', 'n_jobs': 4, 'random_state': 42,
        'max_depth': 30, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300},
    4: {'max_features': 'sqrt', 'n_jobs': 4, 'random_state': 42,
        'max_depth': 40, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300},
}

# Apply the threshold to the predicted probability for positive class 1.
threshold_by_n = {
    2: 0.51,
    3: 0.45712707519321166,
    4: 0.5322798069316441,
}

def load_split(data_dir):
    '''Load paired CSR feature batches and label arrays without densifying X.'''
    feature_files = {p.stem.removeprefix('X_batch_'): p for p in data_dir.glob('X_batch_*.npz')}
    label_files = {p.stem.removeprefix('y_batch_'): p for p in data_dir.glob('y_batch_*.npy')}
    if not feature_files or not label_files:
        raise FileNotFoundError(f'Expected X_batch_*.npz and y_batch_*.npy files in {data_dir}')
    if feature_files.keys() != label_files.keys():
        raise ValueError(
            f'Unpaired batches in {data_dir}; feature IDs={sorted(feature_files)}, '
            f'label IDs={sorted(label_files)}'
        )

    X_parts, y_parts = [], []
    n_features = None
    for batch_id in sorted(feature_files):
        X_batch = sp.load_npz(feature_files[batch_id]).tocsr()
        y_batch = np.load(label_files[batch_id], allow_pickle=False).reshape(-1)
        if X_batch.shape[0] != y_batch.size:
            raise ValueError(
                f'Batch {batch_id} row mismatch in {data_dir}: '
                f'{X_batch.shape[0]} features vs {y_batch.size} labels'
            )
        if n_features is not None and X_batch.shape[1] != n_features:
            raise ValueError(f'Inconsistent feature count in {data_dir}, batch {batch_id}')
        n_features = X_batch.shape[1]
        X_parts.append(X_batch)
        y_parts.append(y_batch)
        print(f'  Loaded batch {batch_id}: X={X_batch.shape}, y={y_batch.shape}')

    X = sp.vstack(X_parts, format='csr')
    y = np.concatenate(y_parts)
    del X_parts, y_parts
    gc.collect()
    return X, y

print(f'Feature data: {DATA_ROOT.resolve()}')
print('Metrics are measured on the held-out test split, not the training rows.')
print('Models are fitted in memory for scoring only; no RF models are saved.')

for n, params in best_params_by_n.items():
    print(f'\n--- {n}-gram ---')
    X_train, y_train = load_split(DATA_ROOT / f'{n}gram' / 'train')
    X_test, y_test = load_split(DATA_ROOT / f'{n}gram' / 'test')
    if X_train.shape[1] != X_test.shape[1]:
        raise ValueError(f'{n}-gram train/test feature counts differ: {X_train.shape[1]} vs {X_test.shape[1]}')
    if not np.array_equal(np.unique(y_train), np.unique(y_test)):
        raise ValueError(f'{n}-gram train/test class labels differ')

    # Fit temporarily to calculate test scores; intentionally do not serialize the model.
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    if len(model.classes_) != 2 or 1 not in model.classes_:
        raise ValueError(f'{n}-gram thresholding expects binary labels including positive class 1; found {model.classes_}')
    positive_index = int(np.flatnonzero(model.classes_ == 1)[0])
    negative_label = model.classes_[model.classes_ != 1][0]
    positive_probabilities = model.predict_proba(X_test)[:, positive_index]
    threshold = threshold_by_n[n]
    y_pred = np.where(positive_probabilities >= threshold, 1, negative_label)
    print(f'Train shape: {X_train.shape}; test shape: {X_test.shape}')
    print(f'Positive-class threshold (label 1): {threshold}')
    print('Accuracy           :', accuracy_score(y_test, y_pred))
    print('Precision (weighted):', precision_score(y_test, y_pred, average='weighted', zero_division=0))
    print('Recall (weighted)   :', recall_score(y_test, y_pred, average='weighted', zero_division=0))
    print('F1 (weighted)       :', f1_score(y_test, y_pred, average='weighted', zero_division=0))
    print('Per-class results:')
    print(classification_report(y_test, y_pred, zero_division=0))

    del model, X_train, y_train, X_test, y_test, y_pred
    gc.collect()


Feature data: D:\BSCS\Cryptojacking Detection Thesis\Graph Aware Cryptojacking Classifier\ml\feature_datasets
Metrics are measured on the held-out test split, not the training rows.
Models are fitted in memory for scoring only; no RF models are saved.

--- 2-gram ---
  Loaded batch 000: X=(100, 76393), y=(100,)
  Loaded batch 001: X=(100, 76393), y=(100,)
  Loaded batch 002: X=(100, 76393), y=(100,)
  Loaded batch 003: X=(100, 76393), y=(100,)
  Loaded batch 004: X=(100, 76393), y=(100,)
  Loaded batch 005: X=(100, 76393), y=(100,)
  Loaded batch 006: X=(100, 76393), y=(100,)
  Loaded batch 007: X=(100, 76393), y=(100,)
  Loaded batch 008: X=(100, 76393), y=(100,)
  Loaded batch 009: X=(100, 76393), y=(100,)
  Loaded batch 010: X=(100, 76393), y=(100,)
  Loaded batch 011: X=(100, 76393), y=(100,)
  Loaded batch 012: X=(100, 76393), y=(100,)
  Loaded batch 013: X=(100, 76393), y=(100,)
  Loaded batch 014: X=(100, 76393), y=(100,)
  Loaded batch 015: X=(100, 76393), y=(100,)
  Loaded bat